In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import math
import glob
from tqdm import tqdm
import pickle
import os
import json

In [2]:
train_df = pd.read_csv("D:/LLM/LAB 1/Dataset/train_combined_ngram.csv")
val_df = pd.read_csv("D:/LLM/LAB 1/Dataset/val_combined_ngram.csv")
test_df = pd.read_csv("D:/LLM/LAB 1/Dataset/test_combined_ngram.csv")
full_df = pd.read_csv("D:/LLM/LAB 1/Dataset/full_data_ngram.csv")

print(f"Full data size: {len(full_df)} Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")
# 

Full data size: 24000 Train size: 19200, Val size: 2400, Test size: 2400


In [3]:
def tokenize(sentence):
    return str(sentence).split()

In [4]:

def tf(sentence, word_to_idx, normalization="unnormalized"):
    tf_value = {}

    tokens = tokenize(sentence)

    for token in tokens:

        idx = word_to_idx.get(token)

        if idx is not None:
            tf_value[idx] = tf_value.get(idx, 0) + 1

    if not tf_value:
        return tf_value


    if normalization == "unnormalized":

        return tf_value

    elif normalization == "total":

        total_words = len(tokens)

        if total_words == 0:
            return {}

        tf_value = {
            idx: freq / total_words
            for idx, freq in tf_value.items()
        }

    elif normalization == "max":

        max_frequency = max(tf_value.values())

        if max_frequency == 0:
            return {}

        tf_value = {
            idx: freq / max_frequency
            for idx, freq in tf_value.items()
        }

    else:
        raise ValueError(
            "normalization must be "
            "'unnormalized', 'total', or 'max'"
        )

    return tf_value


In [5]:
def idf(train_df, word_to_idx):


    N = len(train_df)

    df_value = np.zeros(
        len(word_to_idx),
        dtype=np.uint32
    )

    for sentence in tqdm(
        train_df,
        total=N,
        desc="Computing IDF"
    ):

        tokens = set(tokenize(sentence))

        for token in tokens:

            idx = word_to_idx.get(token)

            if idx is not None:
                df_value[idx] += 1

    idf_values = np.log(
        (N + 1) / (df_value + 1)
    ) + 1

    return idf_values


def normalize_idf(idf_values):

    max_idf = np.max(idf_values)

    if max_idf == 0:
        return idf_values

    normalized_idf = idf_values / max_idf

    return normalized_idf

In [6]:
def calculate_tfidf(sentences,idf_values,word_to_idx,tf_normalization="unnormalized"):

    tfidf_matrix = {}

    for doc_idx, sentence in tqdm(enumerate(sentences),total=len(sentences),desc=f"Computing TF-IDF ({tf_normalization})"):


        tf_values = tf(
            sentence,
            word_to_idx,
            normalization=tf_normalization
        )

        result = {}

        for idx, tf_value in tf_values.items():

            tfidf_value = (
                tf_value * idf_values[idx]
            )

            if tfidf_value != 0:

                result[idx] = tfidf_value

        tfidf_matrix[doc_idx] = result

    return tfidf_matrix


In [7]:

to_work_on = [train_df,val_df,test_df]
vocab_folder = r"D:/LLM/LAB 1/Dataset/vocab"
tfidf_folder = r"D:/LLM/LAB 2/tfidf"

In [8]:
vocab_path = "D:/LLM/LAB 1/Dataset/vocab/metadata/language_vocab_freq_ngram.pkl"



In [9]:
tf_types = ["unnormalized","total","max"]
idf_types = ["unnormalized","normalized"]


In [12]:
print("\n" + "=" * 80)
print("=" * 80)

with open(vocab_path, "rb") as f:
        data = pickle.load(f)

all_words = (
    word
    for lang_vocab in data.values()
    for word in lang_vocab
)

unique_words = set(all_words)


word_to_idx = {
        word: idx
        for idx, word in enumerate(unique_words)
    }


print(f"Vocabulary size: {len(word_to_idx)}")



print("\nComputing unnormalized IDF...")

idf_values = idf(
        full_df["text"],
        word_to_idx
    )

print("\nComputing normalized IDF...")

normalized_idf_values = normalize_idf(
        idf_values
    )

vocab_save_folder = os.path.join(
        tfidf_folder,
        "full vocab"
    )

os.makedirs(
        vocab_save_folder,
        exist_ok=True
    )


with open(
    os.path.join(
            vocab_save_folder,
            "word_to_idx.pkl"
        ),
    "wb"
) as file:

        pickle.dump(
            word_to_idx,
            file,
            protocol=pickle.HIGHEST_PROTOCOL
        )


with open(
    os.path.join(
            vocab_save_folder,
            "idf_unnormalized.pkl"
        ),
        "wb"
) as file:

        pickle.dump(
            idf_values,
            file,
            protocol=pickle.HIGHEST_PROTOCOL
        )


with open(
    os.path.join(
            vocab_save_folder,
            "idf_normalized.pkl"
        ),
        "wb"
) as file:

        pickle.dump(
            normalized_idf_values,
            file,
            protocol=pickle.HIGHEST_PROTOCOL
        )



for tf_type in tf_types:

        for idf_type in idf_types:

            print("\n" + "-" * 80)

            print(
                f"TF normalization : {tf_type}"
            )

            print(
                f"IDF normalization: {idf_type}"
            )


            if idf_type == "unnormalized":
                current_idf = idf_values

            elif idf_type == "normalized":
                current_idf = normalized_idf_values

            combination_folder = (f"tf_{tf_type}_"f"idf_{idf_type}")


            output_folder = os.path.join(
                vocab_save_folder,
                combination_folder
            )


            os.makedirs(
                output_folder,
                exist_ok=True
            )

            output_paths = [

                os.path.join(
                    output_folder,
                    "train_df.csv"
                ),

                os.path.join(
                    output_folder,
                    "val_df.csv"
                ),

                os.path.join(
                    output_folder,
                    "test_df.csv"
                )

            ]


            for df, output_path in zip(
                to_work_on,
                output_paths
            ):

                print(
                    f"\nProcessing {os.path.basename(output_path)}"
                )



                X = calculate_tfidf(
                    df["text"],
                    current_idf,
                    word_to_idx,
                    tf_normalization=tf_type
                )


                new_df = df.copy()

                new_df["text"] = [
                    json.dumps(vector)
                    for vector in X.values()
                ]


                new_df.to_csv(
                    output_path,
                    index=False
                )


                del new_df


                print(
                    f"Saved: {output_path}"
                )


            print(
                f"Finished combination: "
                f"TF={tf_type}, IDF={idf_type}"
            )

print(
        f"\nFinished vocabulary: full vocab"
    )


print("\n" + "=" * 80)
print("ALL TF-IDF COMBINATIONS COMPLETED")
print("=" * 80)


Vocabulary size: 457668

Computing unnormalized IDF...


Computing IDF: 100%|██████████| 24000/24000 [00:04<00:00, 5475.21it/s]



Computing normalized IDF...

--------------------------------------------------------------------------------
TF normalization : unnormalized
IDF normalization: unnormalized

Processing train_df.csv


Computing TF-IDF (unnormalized): 100%|██████████| 19200/19200 [00:04<00:00, 4114.85it/s]


Saved: D:/LLM/LAB 2/tfidf\full vocab\tf_unnormalized_idf_unnormalized\train_df.csv

Processing val_df.csv


...

Saved: D:/LLM/LAB 2/tfidf\full vocab\tf_max_idf_normalized\train_df.csv

Processing val_df.csv


Computing TF-IDF (max): 100%|██████████| 2400/2400 [00:00<00:00, 4178.67it/s]


Saved: D:/LLM/LAB 2/tfidf\full vocab\tf_max_idf_normalized\val_df.csv

Processing test_df.csv


Computing TF-IDF (max): 100%|██████████| 2400/2400 [00:00<00:00, 4660.81it/s]


Saved: D:/LLM/LAB 2/tfidf\full vocab\tf_max_idf_normalized\test_df.csv
Finished combination: TF=max, IDF=normalized

Finished vocabulary: full vocab

ALL TF-IDF COMBINATIONS COMPLETED
